# LeetCode #1145 — Binary Tree Coloring Game (Trò chơi tô màu cây nhị phân)

**Medium · Python 3 · Binary Tree · DFS · Game Theory · Subtree Size**  
[Đề gốc LeetCode](https://leetcode.com/problems/binary-tree-coloring-game/) · [Đối chiếu mô tả và ví dụ](https://leetcode.doocs.org/en/lc/1145/)

```python
class Solution:
    def btreeGameWinningMove(self, root: Optional[TreeNode], n: int, x: int) -> bool:
        ...
```

**Câu hỏi xuyên suốt:** Vì sao chỉ cần đếm 3 vùng quanh nút `x`, thay vì mô phỏng hàng loạt lượt đi của hai người?

## 1. Mục tiêu học tập

1. Hiểu cây nhị phân, `root`, `parent`, cây con và nút kề.
2. Biết vì sao xóa một nút của cây sẽ tách cây thành tối đa **3 thành phần**.
3. Giải thích trò chơi qua **số nút trong các vùng**, không đoán mò nước đi.
4. Dùng **DFS đệ quy** để đếm cây con, nhận diện thời điểm ghé qua `x`.
5. Viết đúng `class Solution`, phân tích `O(n)`, kiểm thử và tự làm 5 bài tập.

## 2. Đề bài, ví dụ và điều kiện

Cây nhị phân có `n` nút, `n` **lẻ**; giá trị của các nút khác nhau trong `1..n`.

- Người thứ nhất chọn nút `x`, tô **đỏ**; người thứ hai (chúng ta) chọn `y != x`, tô **xanh**.
- Từ đó hai người luân phiên: mỗi lượt chỉ được tô một **nút chưa tô nằm kề nút đã tô màu mình**. Kề nghĩa là nối bởi một cạnh cha–con.
- Nếu không có nước hợp lệ thì người đó bỏ lượt. Khi cả hai không thể đi, màu có **nhiều nút** hơn thắng.
- Hỏi có thể **chọn `y` để bảo đảm người xanh thắng** khi hai bên chơi tối ưu không? Trả `True/False`.

| Ví dụ | Dữ liệu | Đáp án |
|---|---|---|
| 1 | `root=[1,2,3,4,5,6,7,8,9,10,11]`, `n=11`, `x=3` | `True` (có thể chọn `y=2`) |
| 2 | `root=[1,2,3]`, `n=3`, `x=1` | `False` |

**Constraints nguồn:** `1 <= x <= n <= 100`, `n` lẻ, tất cả giá trị khác nhau. Trò chơi **không cho tô lại nút đã có màu**.

## 3. Python nhập môn: đối tượng `TreeNode`, `None`, `.left`, `.right`

Một nút lưu `val` (giá trị), `left` (con trái), `right` (con phải). Nếu thiếu con, Python dùng `None`.

Ví dụ cây ba nút:

```text
    1
   / \
  2   3
```

`root.left.val` là `2`, nhưng `root.left.left is None` là `True`.

In [1]:
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"

demo_root = TreeNode(1, TreeNode(2), TreeNode(3))
print('Gốc:', demo_root.val)
print('Con trái:', demo_root.left.val)
print('Có cháu bên trái?', demo_root.left.left is not None)
assert demo_root.left.left is None

Gốc: 1
Con trái: 2
Có cháu bên trái? False


## 4. Cấu trúc cây theo level-order của LeetCode

Trong mảng `[1,2,3,4,5,6,7]`, các vị trí lần lượt được đọc **theo tầng**, trái rồi phải. `None` là vị trí không có nút; ta không thể luôn lấy `2*i+1` khi danh sách có nút rỗng.

`deque` (double-ended queue) cho phép `popleft()` lấy nút phía trước. Hàm dựng cây dưới đây chỉ để học và kiểm thử; **LeetCode đã truyền sẵn `root`**.

In [2]:
def build_tree(level):
    if not level or level[0] is None:
        return None
    root = TreeNode(level[0])
    q = deque([root])
    i = 1
    while q and i < len(level):
        parent = q.popleft()
        if i < len(level) and level[i] is not None:
            parent.left = TreeNode(level[i])
            q.append(parent.left)
        i += 1
        if i < len(level) and level[i] is not None:
            parent.right = TreeNode(level[i])
            q.append(parent.right)
        i += 1
    return root

tree11 = build_tree(list(range(1, 12)))
print('Gốc:', tree11.val, '| con trái:', tree11.left.val, '| con phải:', tree11.right.val)
print('Cháu:', tree11.left.left.val, tree11.left.right.val)
assert tree11.left.right.right.val == 11

Gốc: 1 | con trái: 2 | con phải: 3
Cháu: 4 5


## 5. Python nền tảng: DFS và đệ quy

**DFS — Depth-First Search**: đi sâu vào một nhánh trước khi quay về nhánh khác. **Đệ quy** là hàm tự gọi chính nó.

Công thức `size(node)` = `1 + size(node.left) + size(node.right)`:

- `node is None` ⇒ cây rỗng có kích thước `0` (**điều kiện dừng**).
- Mỗi nút được tính đúng một lần, vì hai cây con không chung nút.

Ví dụ gốc `1` với hai con lá `2,3`: `size(2)=1`, `size(3)=1`, nên `size(1)=3`.

In [3]:
def count_nodes(node):
    if node is None:
        return 0
    left = count_nodes(node.left)
    right = count_nodes(node.right)
    return 1 + left + right

print('Kích thước cây [1,2,3]:', count_nodes(demo_root))
print('Kích thước cây 11 nút:', count_nodes(tree11))
assert count_nodes(demo_root) == 3
assert count_nodes(tree11) == 11

Kích thước cây [1,2,3]: 3
Kích thước cây 11 nút: 11


## 6. Tư duy đầu tiên: mô phỏng tô màu từng lượt?

Có thể thử mọi `y`, rồi xét mọi lựa chọn của đỏ và xanh (backtracking/minimax). Nhưng số chuỗi lượt chơi quá lớn và code phức tạp.

**Chỗ cần hỏi:** Khi `x` đã bị tô đỏ, xanh có thể đi xuyên qua `x` để sang vùng đối diện không? **Không.** Muốn đi qua đó phải tô nút `x` thành xanh, điều luật cấm.

Vì thế ta tìm cách đánh giá vùng mà xanh **chiếm độc quyền** nếu chiếm một nút đứng cạnh `x` ngay từ đầu.

In [4]:
x = 3
print('Trong cây đủ 11 nút, x =', x)
print('Láng giềng của x gồm con trái=6, con phải=7 và cha=1.')
print('Nếu lấy y=2 (bên cha của x), vùng xanh không cần đi qua x.')

Trong cây đủ 11 nút, x = 3
Láng giềng của x gồm con trái=6, con phải=7 và cha=1.
Nếu lấy y=2 (bên cha của x), vùng xanh không cần đi qua x.


## 7. Điểm đột phá: bỏ `x` ra sẽ có tối đa 3 vùng

Giả sử `x` có:

- `L` nút trong **cây con trái**.
- `R` nút trong **cây con phải**.
- `P` nút ở phía còn lại của cây (gồm **tổ tiên và các nhánh khác**).

Mỗi nút khác `x` nằm trong đúng một vùng, nên:

\[
P = n - 1 - L - R.
\]

```text
   [vùng P] ---- x ---- [vùng L]
                  \\
                   [vùng R]
```

Sơ đồ là các **vùng** sau khi bỏ `x`, không phải 3 nút đơn lẻ.

**Đừng nhầm `P` với số tổ tiên:** `P` chứa tất cả nút *không nằm dưới x* nữa.

In [5]:
n, L, R = 11, 1, 1   # Với x=3 trong cây đủ 11 nút
P = n - 1 - L - R
print('L =', L, '| R =', R, '| P =', P, '| tổng + x =', L + R + P + 1)
assert (L, R, P) == (1, 1, 8)

L = 1 | R = 1 | P = 8 | tổng + x = 11


## 8. Vì sao vùng có **hơn n/2** nút là đủ thắng?

Nếu vùng `L` có đa số nút, xanh chọn `y = x.left`. Nếu `R` lớn nhất, chọn `x.right`. Nếu `P` lớn nhất, chọn **cha của `x`**.

Sau lượt chọn đó, đường từ vùng xanh sang phần còn lại **phải qua `x` đã tô đỏ**, nên đỏ không thể vào giành vùng xanh. Xanh có thể từng bước tô hết vùng này (mỗi cây liên thông đều có nút biên chưa tô cho tới khi đã tô hết).

Nếu vùng lớn hơn `n/2`, xanh chắc chắn tô được **hơn một nửa nút**, còn đỏ không thể vượt quá phần còn lại ⇒ xanh thắng.

**Chiều ngược lại:** Nếu cả ba vùng đều không quá `n//2`, bất kể xanh bắt đầu ở đâu, đỏ từ `x` có thể chặn xanh tại `x`, rồi tô các vùng còn lại; xanh không đủ quá nửa nút để thắng.

In [6]:
def winning_region(n, L, R):
    P = n - 1 - L - R
    return max(L, R, P) > n // 2

for case in [(11,1,1), (3,1,1), (5,2,2), (5,0,0), (7,1,3)]:
    n,L,R = case
    print(f'n={n}, L={L}, R={R}, P={n-1-L-R}, win={winning_region(n,L,R)}')
assert winning_region(11,1,1) is True
assert winning_region(3,1,1) is False

n=11, L=1, R=1, P=8, win=True
n=3, L=1, R=1, P=0, win=False
n=5, L=2, R=2, P=0, win=False
n=5, L=0, R=0, P=4, win=True
n=7, L=1, R=3, P=2, win=False


## 9. Bước thu thập thông tin: tìm đúng nút có giá trị `x`

Giá trị nút **không phải chỉ số mảng**; ta phải duyệt để tìm nút có `.val == x`.

Trước tiên viết `find_node` bằng đệ quy; `return find_node(left) or find_node(right)` có nghĩa: thử trái; nếu là `None` thì thử phải.

In [7]:
def find_node(root, x):
    if root is None or root.val == x:
        return root
    return find_node(root.left, x) or find_node(root.right, x)

node_x = find_node(tree11, 3)
print('Nút x:', node_x)
print('Kích thước L, R:', count_nodes(node_x.left), count_nodes(node_x.right))
assert (count_nodes(node_x.left), count_nodes(node_x.right)) == (1, 1)

Nút x: TreeNode(3)
Kích thước L, R: 1 1


## 10. Lời giải dễ nghĩ: tìm `x` rồi đếm hai cây con

Chạy `find_node` tìm nút, sau đó gọi `count_nodes` trên con trái và con phải. Tổng `O(n)` vì mỗi bước quét không lặp nhau theo cấp độ toàn cây.

Code ngắn, có lợi cho người mới. Phiên bản sau sẽ **gộp thành đúng một DFS**.

In [8]:
def can_blue_win_two_pass(root, n, x):
    target = find_node(root, x)
    left_count = count_nodes(target.left)
    right_count = count_nodes(target.right)
    parent_count = n - 1 - left_count - right_count
    return max(left_count, right_count, parent_count) > n // 2

print('Ví dụ 1:', can_blue_win_two_pass(tree11, 11, 3))
print('Ví dụ 2:', can_blue_win_two_pass(build_tree([1,2,3]), 3, 1))
assert can_blue_win_two_pass(tree11, 11, 3) is True
assert can_blue_win_two_pass(build_tree([1,2,3]), 3, 1) is False

Ví dụ 1: True
Ví dụ 2: False


## 11. Cải tiến: DFS một lần, lúc quay lui ghi nhận `L`, `R`

Khi hàm DFS đang ở nút `x`, nó vừa tính xong `left_size` và `right_size`. Đây đúng là hai số ta cần!

- `nonlocal left_size, right_size`: cho phép hàm trong cập nhật biến ở hàm ngoài.
- `if node.val == x`: ghi nhận hai kết quả đã tính.
- Hàm `dfs` **vẫn trả về kích thước của cả cây con** để phép tính ở nút cha chính xác.

Tránh đếm lại hai cây con sau khi tìm `x`.

In [9]:
def can_blue_win_one_dfs(root, n, x):
    left_size = right_size = 0

    def dfs(node):
        nonlocal left_size, right_size
        if node is None:
            return 0
        L = dfs(node.left)
        R = dfs(node.right)
        if node.val == x:
            left_size, right_size = L, R
        return 1 + L + R

    dfs(root)
    parent_size = n - 1 - left_size - right_size
    return max(left_size, right_size, parent_size) > n // 2

print('DFS một lượt:', can_blue_win_one_dfs(tree11, 11, 3))
assert can_blue_win_one_dfs(tree11, 11, 3)

DFS một lượt: True


## 12. Mô phỏng DFS trên cây `[1,2,3,4,5,6,7]`, `x=2`

```text
        1
      /   \\
     2     3
    / \\   / \\
   4   5 6   7
```

Khi DFS quay lui tại nút `2`: `L=1`, `R=1`, `P=7-1-1-1=4`.

Vì `4 > 7//2 = 3`, xanh chọn nút `1`, chiếm được phía ngoài cây con của `2`.

| Trạng thái | Giá trị |
|---|---:|
| `dfs(4)`, `dfs(5)` | `1`, `1` |
| `dfs(2)` | `1+1+1=3` |
| `L, R` khi gặp `x=2` | `1, 1` |
| `P` | `4` |
| Có vùng hơn một nửa? | Có (`4>3`) |

In [10]:
def trace_coloring(root, n, x):
    sizes = {}
    def dfs(node):
        if node is None:
            return 0
        left = dfs(node.left)
        right = dfs(node.right)
        sizes[node.val] = (left, right, left + right + 1)
        print(f'Quay lui nút {node.val:>2}: trái={left}, phải={right}, tổng={1+left+right}')
        return 1 + left + right
    dfs(root)
    L,R,_ = sizes[x]
    P = n - 1 - L - R
    print(f'\nx={x} => L={L}, R={R}, P={P}; ngưỡng={n//2}; BLUE WIN={max(L,R,P)>n//2}')

trace_coloring(build_tree([1,2,3,4,5,6,7]), 7, 2)

Quay lui nút  4: trái=0, phải=0, tổng=1
Quay lui nút  5: trái=0, phải=0, tổng=1
Quay lui nút  2: trái=1, phải=1, tổng=3
Quay lui nút  6: trái=0, phải=0, tổng=1
Quay lui nút  7: trái=0, phải=0, tổng=1
Quay lui nút  3: trái=1, phải=1, tổng=3
Quay lui nút  1: trái=3, phải=3, tổng=7

x=2 => L=1, R=1, P=4; ngưỡng=3; BLUE WIN=True


## 13. Code nộp LeetCode — chính xác chữ ký

LeetCode có sẵn định nghĩa `TreeNode`. Code dưới đây dùng `Optional` từ `typing` (đã import đầu notebook). Không có `print` trong giải pháp nộp.

In [11]:
class Solution:
    def btreeGameWinningMove(self, root: Optional[TreeNode], n: int, x: int) -> bool:
        left_size = right_size = 0

        def dfs(node):
            nonlocal left_size, right_size
            if node is None:
                return 0
            L = dfs(node.left)
            R = dfs(node.right)
            if node.val == x:
                left_size, right_size = L, R
            return L + R + 1

        dfs(root)
        parent_size = n - left_size - right_size - 1
        return max(left_size, right_size, parent_size) > n // 2

solver = Solution()
assert solver.btreeGameWinningMove(tree11, 11, 3) is True
assert solver.btreeGameWinningMove(build_tree([1,2,3]), 3, 1) is False
print('2 ví dụ đề: PASS')

2 ví dụ đề: PASS


## 14. Kiểm thử biên và phản ví dụ

- `n=1`: người xanh không có `y` hợp lệ; trong bài thường `n` là số lẻ và có thể bằng 1, kết quả không thể thắng.
- `x` là **lá**: `L=R=0`, toàn bộ `n-1` nút ở vùng cha; với `n>=3` xanh thắng.
- `x` ở **gốc** và hai cây con không quá nửa: xanh không thắng.
- Cây lệch sang một bên vẫn phải tính đúng phía cha.

In [12]:
edge_cases = [
    ([1],1,1,False),
    ([1,2,3],3,2,True),
    ([1,2,3],3,3,True),
    ([1,2,3],3,1,False),
    ([1,2,3,4,5],5,1,True),
    ([1,2,3,4,5],5,2,False),
    ([1,2,None,3,None,4,None,5],5,3,False),
]
for level,n,x,expected in edge_cases:
    root=build_tree(level)
    got=solver.btreeGameWinningMove(root,n,x)
    assert got==expected,(level,n,x,got,expected)
print('7 ca biên: PASS')

7 ca biên: PASS


## 15. Vì sao đúng? (Chứng minh bằng cấu trúc cây)

**Bất biến 1:** `dfs(node)` luôn trả số nút trong cây con gốc `node`. Điều kiện dừng đúng cho cây rỗng; bước đệ quy đúng vì các cây con rời nhau.

**Bất biến 2:** Khi gặp `x`, hai số `L,R` đã là kích thước chính xác của hai vùng bên dưới. Vùng thứ ba có `P=n-1-L-R` nút.

**Chiến lược:** Nếu một vùng có > `n/2` nút, xanh tô điểm giáp `x` trong vùng đó, cô lập nó khỏi đỏ; đỏ không thể xuyên qua `x` để cạnh tranh. Nếu không có vùng như vậy, vì mọi đường từ vùng xanh đến các vùng khác đều qua `x`, xanh không chiếm được quá `n/2` nút.

Đây là **lý do tổng quát**, không phụ thuộc hình dáng cây hay giá trị cụ thể tại các nút.

## 16. Độ phức tạp

Gọi `n` là số nút, `h` là chiều cao cây:

| Cách làm | Time | Space phụ |
|---|---|---|
| Tìm `x` + đếm hai cây con | `O(n)` | `O(h)` stack đệ quy |
| Một DFS ghi `L,R` | **`O(n)`** | **`O(h)`** stack đệ quy |

Tệ nhất cây là một dây dài ⇒ `h=n`, nên bộ nhớ phụ `O(n)` worst-case. Với cây cân bằng `h=O(log n)`.

Đề `n<=100` nên recursion Python an toàn; không có thao tác mảng sao chép.

## 17. Một cách kiểm tra độc lập: dựng đồ thị rồi bỏ `x`

Cây là đồ thị vô hướng (nút cha và con là hai đỉnh kề nhau). Nếu **xóa `x` và mọi cạnh dính vào `x`**, ta tìm các thành phần liên thông còn lại bằng BFS. Kích thước lớn nhất cần bằng `max(L,R,P)`.

Cách này dài hơn và tốn bộ nhớ, nhưng hữu ích để **đối chiếu ngẫu nhiên** với code DFS.

In [13]:
def winning_by_components(root, n, x):
    adj = {i: [] for i in range(1, n+1)}
    q = [root]
    while q:
        node = q.pop()
        if node is None:
            continue
        for child in (node.left, node.right):
            if child is not None:
                adj[node.val].append(child.val)
                adj[child.val].append(node.val)
                q.append(child)
    seen={x}
    largest=0
    for start in range(1, n+1):
        if start in seen:
            continue
        todo=[start]
        seen.add(start)
        count=0
        while todo:
            u=todo.pop()
            count+=1
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    todo.append(v)
        largest=max(largest,count)
    return largest > n//2

assert winning_by_components(tree11,11,3) == solver.btreeGameWinningMove(tree11,11,3)
print('Kiểm tra bằng thành phần liên thông: PASS')

Kiểm tra bằng thành phần liên thông: PASS


## 18. Đối chiếu trên cây ngẫu nhiên: tránh chỉ tin vài ví dụ

Ta dựng cây nhị phân ngẫu nhiên với mỗi nút `1..n` duy nhất, `n` lẻ. So sánh **DFS đếm cây con** với **BFS thành phần liên thông** cho **mọi `x`**.

In [14]:
import random
rng = random.Random(1145)

def random_binary_tree(n):
    nodes={v:TreeNode(v) for v in range(1,n+1)}
    for v in range(2,n+1):
        candidates=[u for u in range(1,v) if nodes[u].left is None or nodes[u].right is None]
        parent=nodes[rng.choice(candidates)]
        if parent.left is None and parent.right is None:
            side=rng.choice((0,1))
        else:
            side=0 if parent.left is None else 1
        if side==0:
            parent.left=nodes[v]
        else:
            parent.right=nodes[v]
    return nodes[1]

checks=0
for n in range(1,24,2):
    for trial in range(12):
        root=random_binary_tree(n)
        for x in range(1,n+1):
            a=solver.btreeGameWinningMove(root,n,x)
            b=winning_by_components(root,n,x)
            assert a==b,(n,x,a,b)
            checks+=1
print('Đã kiểm tra',checks,'trường hợp DFS/BFS: PASS')

Đã kiểm tra 1728 trường hợp DFS/BFS: PASS


## 19. Lỗi thường gặp và cách tránh

| Lỗi | Ví dụ sai | Cách sửa |
|---|---|---|
| Quên nút `x` | `P=n-L-R` | `P=n-1-L-R` |
| Dùng `>=` | `max_region >= n//2` | Cần **hơn** nửa: `> n//2` |
| Chỉ tính 2 cây con | `max(L,R)` | Phải xét cả vùng **cha `P`** |
| Hiểu `P` chỉ là số tổ tiên | Đếm parent đến root | Gồm mọi nhánh *ngoài cây con của x* |
| Quên điều kiện dừng | Gọi `node.left` khi `None` | `if node is None: return 0` |

Ví dụ `n=3, x=1` có `L=R=1`, không vùng nào >1.5 ⇒ **False**, không được trả True vì «có hai vùng».

In [15]:
n,L,R=3,1,0
correct=max(L,R,n-1-L-R)>n//2
mistaken=(max(L,R,n-L-R)>n//2)  # cố ý sai: quên trừ nút x
print('Đúng:', correct, '| Sai do quên trừ nút x:', mistaken)
assert correct is False and mistaken is True

Đúng: False | Sai do quên trừ nút x: True


## 20. Nhận dạng pattern và biến thể

- Đề nói «bỏ một đỉnh khỏi cây» ⇒ nghĩ tới **các thành phần liên thông**.
- Đề nói «nửa số đỉnh» ⇒ so sánh kích thước lớn nhất với `n//2`.
- Đề cần thông tin kích thước từng cây con ⇒ **DFS hậu tự (postorder)**: tính trái/phải **trước** nút hiện tại.
- Biến thể: đếm tất cả `x` mà người thứ hai thắng; học cách tái sử dụng `subtree_size`.

Sự khác biệt với **minimax thuần túy**: ta dùng một tính chất riêng của **cây** để thay thế tìm kiếm trạng thái chơi phức tạp.

In [16]:
root7=build_tree([1,2,3,4,5,6,7])
outcome={x:solver.btreeGameWinningMove(root7,7,x) for x in range(1,8)}
print('Người xanh có thể thắng theo từng x:',outcome)
assert outcome[1] is False and outcome[4] is True

Người xanh có thể thắng theo từng x: {1: False, 2: True, 3: True, 4: True, 5: True, 6: True, 7: True}


## 21. Tự luyện — 5 bài, tăng dần độ khó

**Bài 1 — Đoán kết quả:** `n=3`, `x` là gốc, mỗi con là lá. Xanh thắng không? Giải thích theo `L,R,P`.

**Bài 2 — Tính tay:** `n=11`, khi tới `x` có `L=3`, `R=1`. Hãy tính `P`, xác định xanh thắng không và nên chọn vùng nào.

**Bài 3 — TODO:** viết hàm `exercise_region_sizes(n,L,R)` trả `(L,R,P)` và kiểm tra `L+R+P+1==n`.

**Bài 4 — TODO:** viết hàm `exercise_count(root)` dùng DFS đếm số nút.

**Bài 5 — Vận dụng:** viết `exercise_winning_x(root,n)` trả danh sách các giá trị `x` cho người xanh thắng (đã sắp tăng).

**Nguyên tắc:** làm trước rồi mới mở phần gợi ý và đáp án ở cuối.

In [17]:
# TODO — chỉ khai báo, notebook vẫn chạy bình thường.
def exercise_region_sizes(n, L, R):
    # TODO: tính P và kiểm tra đẳng thức tổng nút
    pass

def exercise_count(root):
    # TODO: DFS với điều kiện dừng None
    pass

def exercise_winning_x(root, n):
    # TODO: duyệt x=1..n và tận dụng Solution
    pass

# Tự mở các assert dưới đây SAU KHI hoàn thành TODO:
# assert exercise_region_sizes(11, 3, 1) == (3, 1, 6)
# assert exercise_count(build_tree([1,2,3])) == 3
# assert exercise_winning_x(build_tree([1,2,3]),3) == [2,3]

## 22. Gợi ý 3 mức

| Bài | Gợi ý 1 — ý tưởng | Gợi ý 2 — công cụ | Gợi ý 3 — gần code |
|---|---|---|---|
| 1 | Chia 3 vùng | `L=R=1` | `P=3-1-1-1=0`, so với `n//2` |
| 2 | Không chỉ nhìn `L,R` | `P=n-1-L-R` | `P=6`, `6>5` ⇒ thắng ở vùng cha |
| 3 | Bảo toàn tổng số nút | `assert` | `P=n-1-L-R` |
| 4 | Đếm các cây con | `if root is None` | `return 1+dfs(left)+dfs(right)` |
| 5 | Thử tất cả x | `range(1,n+1)` | `if Solution().btreeGameWinningMove(root,n,x): append(x)` |

## 23. Đáp án tham khảo (xem sau khi tự làm)

**Bài 1:** `False`, vì `L=1,R=1,P=0`, `max=1` không lớn hơn `n//2=1`.

**Bài 2:** `P=11-1-3-1=6`. Xanh chọn nút phía cha của `x`, đảm bảo chiếm vùng 6/11 ⇒ `True`.

Đáp án code bài 3–5 trong cell kế tiếp.

In [18]:
def answer_region_sizes(n,L,R):
    P=n-1-L-R
    assert L+R+P+1==n
    return L,R,P

def answer_count(root):
    if root is None:
        return 0
    return 1+answer_count(root.left)+answer_count(root.right)

def answer_winning_x(root,n):
    sol=Solution()
    return [x for x in range(1,n+1) if sol.btreeGameWinningMove(root,n,x)]

assert answer_region_sizes(11,3,1)==(3,1,6)
assert answer_count(build_tree([1,2,3]))==3
assert answer_winning_x(build_tree([1,2,3]),3)==[2,3]
print('Đáp án TODO: PASS')

Đáp án TODO: PASS


## 24. Tổng kết và checklist nộp bài

```text
DFS hậu tự (trái, phải, nút):
    tại x: L = size(x.left), R = size(x.right)
P = n - 1 - L - R
answer = max(L,R,P) > n//2
```

- [ ] Hiểu vì sao cây bị chia thành **tối đa 3 vùng**.
- [ ] Có đủ `L,R,P` và trừ `1` cho `x`.
- [ ] Dùng so sánh **nghiêm ngặt `>`**; đáp án kiểu `bool`.
- [ ] `dfs(None)==0`; trả `1+L+R` đúng ở mọi nút.
- [ ] `Solution.btreeGameWinningMove(root,n,x)` đúng chữ ký; không `print` khi submit.
- [ ] `O(n)` thời gian, `O(h)` stack; đã thử gốc, lá, cây lệch.